In [1]:

import os
os.environ["GIT_PYTHON_REFRESH"] = "quiet"
import git

from myosuite.utils import gym
import skvideo
skvideo.setFFmpegPath(r"C:\Users\rohan\Downloads\ffmpeg-2025-08-18-git-0226b6fb2c-essentials_build\ffmpeg-2025-08-18-git-0226b6fb2c-essentials_build\bin")
import skvideo.io

import numpy as np
import os

MyoSuite:> Registering Myo Envs


c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment myoArmReachFixed-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment myoSarcArmReachFixed-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment myoFatiArmReachFixed-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment myoArmReachRandom-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registr

In [1]:
# Strictly data_preprocessing-style evaluation loop, but only record first and last episode video
import myosuite
from myosuite.utils import gym
import skvideo
import skvideo.io
import numpy as np
import os
from stable_baselines3 import PPO
import mujoco
import matplotlib.pyplot as plt
import time
from IPython.display import HTML
from base64 import b64encode

def show_video(video_path, video_width=400):
    video_file = open(video_path, "r+b").read()
    video_url = f'data:video/mp4;base64,{b64encode(video_file).decode()}'
    return HTML(f"<video autoplay width={video_width} controls><source src=\"{video_url}\"></video>")

import PIL.Image, PIL.ImageDraw, PIL.ImageFont
def add_text_to_frame(frame, text, pos=(20, 20), color=(255, 0, 0), fontsize=12):
    if isinstance(frame, np.ndarray):
        frame = PIL.Image.fromarray(frame)
    draw = PIL.ImageDraw.Draw(frame)
    draw.text(pos, text, fill=color)
    return frame

env_name = "myoElbowPose1D6MRandom-v0"
GENERATE_VIDEO = True
n_eps = 250
env = gym.make(env_name)
policy_path = 'ElbowPose_policy'
model = PPO.load(policy_path)
env.unwrapped.set_fatigue_reset_random(False)
env.reset(fatigue_reset=True)

env.unwrapped.muscle_fatigue.MA[:] = 0.0   # no active units
env.unwrapped.muscle_fatigue.MR[:] = 0.0   # no resting units
env.unwrapped.muscle_fatigue.MF[:] = 1.0   # 100% fatigued
   # all fully fatigued

#print("MA:", env.unwrapped.muscle_fatigue.MA)
#print("MR:", env.unwrapped.muscle_fatigue.MR)
#print("MF:", env.unwrapped.muscle_fatigue.MF)  # should be all ones


env.unwrapped.sim.model.cam_poscom0[0]= np.array([-1.3955, -0.3287,  0.6579])
max_steps = env.spec.max_episode_steps
data_store = []
frames_first = []
frames_last = []
video_path_first = None
video_path_last = None

low = env.unwrapped.target_jnt_range[:, 0]
high = env.unwrapped.target_jnt_range[:, 1]
new_target = np.random.uniform(low, high)
env.unwrapped.target_jnt_value = [1.7]
env.unwrapped.target_type = 'fixed'
env.unwrapped.update_target(restore_sim=True)

start_time = time.time()
for ep in range(n_eps):
    env.reset()
    #print(f"Start of episode {ep+1}: MF={env.unwrapped.muscle_fatigue.MF}, qpos={env.unwrapped.sim.data.qpos}")
    #env.reset(fatigue_reset=False)
    #env.unwrapped.target_jnt_value = [np.deg2rad(80)]
    #env.unwrapped.target_type = 'fixed'
    #env.unwrapped.weight_range=(0,0)
    #env.unwrapped.update_target(restore_sim=False) 
    print(f"Ep {ep+1} of {n_eps}")
    for _cstep in range(env.spec.max_episode_steps):
        #env.mj_render()
        record_first = GENERATE_VIDEO and (ep == 0)
        record_last = GENERATE_VIDEO and (ep == n_eps - 1)
        if record_first or record_last:
            frame = env.unwrapped.sim.renderer.render_offscreen(width=400, height=400, camera_id=0)
            _current_time = (ep*env.spec.max_episode_steps + _cstep)*env.unwrapped.dt
            frame = np.array(add_text_to_frame(frame,
                    f"t={str(int(_current_time//60)).zfill(2)}:{str(int(_current_time%60)).zfill(2)}min",
                    pos=(285, 3), color=(0, 0, 0), fontsize=18))
            if record_first:
                frames_first.append(frame)
            if record_last:
                frames_last.append(frame)
        o = env.unwrapped.get_obs()
        a = model.predict(o)[0]
        next_o, r, done, _, ifo = env.step(a)
        data_store.append({"action":a.copy(), 
                            "jpos":env.unwrapped.sim.data.qpos.copy(), 
                            "mlen":env.unwrapped.sim.data.actuator_length.copy(), 
                            "act":env.unwrapped.sim.data.act.copy(),
                            "reward":r,
                            "solved":env.unwrapped.rwd_dict['solved'].item(),
                            "pose_err":env.unwrapped.get_obs_dict(env.unwrapped.sim)["pose_err"]
        })
                            #"MA":env.unwrapped.muscle_fatigue.MA.copy(),
                            #"MR":env.unwrapped.muscle_fatigue.MR.copy(),
                            #"MF":env.unwrapped.muscle_fatigue.MF.copy(),
                            #"ctrl":env.unwrapped.last_ctrl.copy()})
        if done:
            break
env.close()

if GENERATE_VIDEO and frames_first:
    os.makedirs('videos', exist_ok=True)
    video_path_first = 'videos/episode_first.mp4'
    skvideo.io.vwrite(video_path_first, np.asarray(frames_first), outputdict={'-pix_fmt': 'yuv420p'})
if GENERATE_VIDEO and frames_last:
    os.makedirs('videos', exist_ok=True)
    video_path_last = 'videos/episode_last.mp4'
    skvideo.io.vwrite(video_path_last, np.asarray(frames_last), outputdict={'-pix_fmt': 'yuv420p'})

if GENERATE_VIDEO and video_path_first:
    display(show_video(video_path_first))
if GENERATE_VIDEO and video_path_last:
    display(show_video(video_path_last))

import pandas as pd
df = pd.DataFrame(data_store)
_env_dt = env.unwrapped.dt

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment myoArmReachFixed-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment myoSarcArmReachFixed-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry."

MyoSuite:> Registering Myo Envs
    MyoSuite: A contact-rich simulation suite for musculoskeletal motor control
        Vittorio Caggiano, Huawei Wang, Guillaume Durandau, Massimo Sartori, Vikash Kumar
        L4DC-2019 | https://sites.google.com/view/myosuite
    


FileNotFoundError: [Errno 2] No such file or directory: 'ElbowPose_policy.zip'

In [ ]:
# --- Reference-style plots from data_preprocessing.ipynb, using collected all_data ---
import matplotlib.pyplot as plt
import numpy as np

# Use in-memory all_data from previous cell
data_store = df.to_dict('records')
_env_dt = env.unwrapped.dt

# Plot MF (Fatigued Motor Units) for each muscle
plt.figure()
for _muscleid in range(len(data_store[0]['MF'])):
    plt.plot(_env_dt*np.arange(len(data_store)), np.array([d['MF'][_muscleid] for d in data_store]), label=f"MF[{_muscleid}]")
plt.legend()
plt.title('Fatigued Motor Units')

# Plot MR (Resting Motor Units) for each muscle
plt.figure()
for _muscleid in range(len(data_store[0]['MR'])):
    plt.plot(_env_dt*np.arange(len(data_store)), np.array([d['MR'][_muscleid] for d in data_store]), label=f"MR[{_muscleid}]")
plt.legend()
plt.title('Resting Motor Units')

# Plot MA (Active Motor Units) for each muscle
plt.figure()
for _muscleid in range(len(data_store[0]['MA'])):
    plt.plot(_env_dt*np.arange(len(data_store)), np.array([d['MA'][_muscleid] for d in data_store]), label=f"MA[{_muscleid}]")
plt.legend()
plt.title('Active Motor Units')

# Plot pose error as a continuous time series
plt.figure()
plt.plot(_env_dt*np.arange(len(data_store)), np.array([np.linalg.norm(d['pose_err']) for d in data_store]))
plt.title('Pose Error')

# Plot reward as a continuous time series
plt.figure()
plt.plot(_env_dt*np.arange(len(data_store)), np.array([d['reward'] for d in data_store]))
plt.title(f"Reward (Total: {np.array([d['reward'] for d in data_store]).sum():.2f})")

# Plot solved if present
if 'solved' in data_store[0]:
    plt.figure()
    solved_arr = np.array([d['solved'] for d in data_store])
    plt.scatter(_env_dt*np.arange(len(data_store))[solved_arr.astype(bool)], solved_arr[solved_arr.astype(bool)])
    plt.title('Success')

# Print muscle fatigue equilibrium if present
if 'MF' in data_store[-1]:
    print(f"Muscle Fatigue Equilibrium: {data_store[-1]['MF']}")



In [2]:
# Strictly data_preprocessing-style evaluation loop for ExoOnlyWrapper, only record first and last episode video
import myosuite
from myosuite.utils import gym
import skvideo.io
import numpy as np
import os
from stable_baselines3 import PPO
import matplotlib.pyplot as plt
import time
from IPython.display import HTML
from base64 import b64encode
import PIL.Image, PIL.ImageDraw

# ------------------------
# Utility: show video inline
# ------------------------125125125111111
def show_video(video_path, video_width=400):
    video_file = open(video_path, "r+b").read()
    video_url = f'data:video/mp4;base64,{b64encode(video_file).decode()}'
    return HTML(f"<video autoplay width={video_width} controls><source src=\"{video_url}\"></video>")

def add_text_to_frame(frame, text, pos=(20, 20), color=(255, 0, 0), fontsize=12):
    if isinstance(frame, np.ndarray):
        frame = PIL.Image.fromarray(frame)
    draw = PIL.ImageDraw.Draw(frame)
    draw.text(pos, text, fill=color)
    return frame

# ------------------------
# Import custom wrapper
# ------------------------
from stable_baselines3 import PPO
import gymnasium as gym

class ExoOnlyWrapper(gym.Env):
    def __init__(self, base_env, frozen_policy_path, smart_reset=False):
        super(ExoOnlyWrapper, self).__init__()
        self.base_env = base_env
        self.frozen_policy = PPO.load(frozen_policy_path)
        self.action_space = gym.spaces.Box(low=np.array([0.0]), high=np.array([1.0]), dtype=np.float32)

        self.n_muscles = len(self.base_env.unwrapped.muscle_fatigue.MA)
        original_obs_space = self.base_env.observation_space
        mf_low = np.zeros(self.n_muscles, dtype=np.float32)
        mf_high = np.ones(self.n_muscles, dtype=np.float32)
        if isinstance(original_obs_space, gym.spaces.Box):
            combined_low = np.concatenate([original_obs_space.low, mf_low])
            combined_high = np.concatenate([original_obs_space.high, mf_high])
            
            self.observation_space = gym.spaces.Box(
                low=combined_low,
                high=combined_high,
                dtype=np.float32
            )
        else:
            raise ValueError("Base environment observation space must be Box type")

        #self.observation_space = self.base_env.observation_space
        self.smart_reset = smart_reset
        if smart_reset == False:
            self.base_env.unwrapped.set_fatigue_reset_random(True)    

    def _get_extended_obs(self, base_obs):
        mf_values = self.base_env.unwrapped.muscle_fatigue.MF.copy().astype(np.float32)
        extended_obs = np.concatenate([base_obs, mf_values])
        
        return extended_obs
    
    def reset(self, **kwargs):
        obs, info = self.base_env.reset(fatigue_reset=True)
        low = self.base_env.unwrapped.target_jnt_range[:, 0]
        high = self.base_env.unwrapped.target_jnt_range[:, 1]
        new_target = np.random.uniform(low, high)
        self.base_env.unwrapped.target_jnt_value = new_target
        self.base_env.unwrapped.target_type = 'fixed'

        if self.smart_reset:
            n_muscles = len(self.base_env.unwrapped.muscle_fatigue.MA)

            # Random fatigue per muscle in [0.7, 1.0]
            MF = np.random.uniform(0.7, 1.0, size=n_muscles)

            # For the remaining fraction (1 - MF), randomly split between MA and MR
            remaining = 1.0 - MF
            split = np.random.uniform(0.0, 1.0, size=n_muscles)  # split ratio
            MA = remaining * split
            MR = remaining * (1.0 - split)

            # Write back into environment
            self.base_env.unwrapped.muscle_fatigue.MA[:] = MA
            self.base_env.unwrapped.muscle_fatigue.MR[:] = MR
            self.base_env.unwrapped.muscle_fatigue.MF[:] = MF

        extended_obs = self._get_extended_obs(obs)

        return extended_obs, info

    def step(self, exo_action):
        exo_action = np.array([exo_action], dtype=np.float32).reshape(-1)
        
        base_obs = self.base_env.unwrapped.get_obs()
        muscle_actions, _ = self.frozen_policy.predict(base_obs, deterministic=True)

        full_action = np.concatenate([exo_action, muscle_actions])
        obs, reward, done, truncated, info = self.base_env.step(full_action)

        extended_obs = self._get_extended_obs(obs)
        return extended_obs, reward, done, truncated, info

    def render(self, *args, **kwargs):
        return self.base_env.render(*args, **kwargs)

    def close(self):
        self.base_env.close



In [13]:

env_name = "myoFatiElbowPose1D6MExoRandom-v0"
GENERATE_VIDEO = True
n_eps = 250

# Load base env and wrap it
base_env = gym.make(env_name)
exo_env = ExoOnlyWrapper(base_env, frozen_policy_path="ElbowPose_policy")

exo_env.base_env.unwrapped.set_fatigue_reset_random(False)
exo_env.base_env.reset()

max_steps = exo_env.base_env.spec.max_episode_steps
data_store = []
frames_first = []
frames_last = []
video_path_first = None
video_path_last = None

low = exo_env.base_env.unwrapped.target_jnt_range[:, 0]
high = exo_env.base_env.unwrapped.target_jnt_range[:, 1]
new_target = np.random.uniform(low, high)
exo_env.base_env.unwrapped.target_jnt_value = [1.7]
exo_env.base_env.unwrapped.target_type = 'fixed'

exo_env.base_env.unwrapped.set_fatigue_reset_random(False)
exo_env.base_env.reset(fatigue_reset=True)

n_muscles = len(exo_env.base_env.unwrapped.muscle_fatigue.MA)

MF = np.random.uniform(0.7, 1.0, size=n_muscles)


remaining = 1.0 - MF
split = np.random.uniform(0.0, 1.0, size=n_muscles)  # split ratio
MA = remaining * split
MR = remaining * (1.0 - split)

exo_env.base_env.unwrapped.muscle_fatigue.MA[:] = MA
exo_env.base_env.unwrapped.muscle_fatigue.MR[:] = MR
exo_env.base_env.unwrapped.muscle_fatigue.MF[:] = MF

print("MA:", exo_env.base_env.unwrapped.muscle_fatigue.MA)
print("MR:", exo_env.base_env.unwrapped.muscle_fatigue.MR)
print("MF:", exo_env.base_env.unwrapped.muscle_fatigue.MF)  # should be all ones

start_time = time.time()
for ep in range(n_eps):
    #exo_env.reset(fatigue_reset=False)
    print(f"Ep {ep+1} of {n_eps}")    
    for _cstep in range(max_steps):
        record_first = GENERATE_VIDEO and (ep == 0)
        record_last = GENERATE_VIDEO and (ep == n_eps - 1)
        if record_first or record_last:
            frame = exo_env.base_env.unwrapped.sim.renderer.render_offscreen(width=400, height=400, camera_id=0)
            _current_time = (ep*max_steps + _cstep)*exo_env.base_env.unwrapped.dt
            frame = np.array(add_text_to_frame(frame,
                    f"t={str(int(_current_time//60)).zfill(2)}:{str(int(_current_time%60)).zfill(2)}min",
                    pos=(285, 3), color=(0, 0, 0), fontsize=18))
            if record_first:
                frames_first.append(frame)
            if record_last:
                frames_last.append(frame)
        o = exo_env.base_env.unwrapped.get_obs()
        a = np.array([0.0], dtype=np.float32)
        next_o, r, done, _, info = exo_env.step(a)
        data_store.append({
            "action": a.copy(),
            "jpos": exo_env.base_env.unwrapped.sim.data.qpos.copy(),
            "mlen": exo_env.base_env.unwrapped.sim.data.actuator_length.copy(),
            "act": exo_env.base_env.unwrapped.sim.data.act.copy(),
            "reward": r,
            "solved": exo_env.base_env.unwrapped.rwd_dict['solved'].item(),
            "pose_err": exo_env.base_env.unwrapped.get_obs_dict(exo_env.base_env.unwrapped.sim)["pose_err"],
            "MA": exo_env.base_env.unwrapped.muscle_fatigue.MA.copy(),
            "MR": exo_env.base_env.unwrapped.muscle_fatigue.MR.copy(),
            "MF": exo_env.base_env.unwrapped.muscle_fatigue.MF.copy(),
            "ctrl": exo_env.base_env.unwrapped.last_ctrl.copy()
        })
        if done:
            break
exo_env.close()


if GENERATE_VIDEO and frames_first:
    os.makedirs('videos', exist_ok=True)
    video_path_first = 'videos/episode_first.mp4'
    skvideo.io.vwrite(video_path_first, np.asarray(frames_first), outputdict={'-pix_fmt': 'yuv420p'})
if GENERATE_VIDEO and frames_last:
    os.makedirs('videos', exist_ok=True)
    video_path_last = 'videos/episode_last.mp4'
    skvideo.io.vwrite(video_path_last, np.asarray(frames_last), outputdict={'-pix_fmt': 'yuv420p'})

if GENERATE_VIDEO and video_path_first:
    display(show_video(video_path_first))
if GENERATE_VIDEO and video_path_last:
    display(show_video(video_path_last))

import pandas as pd
df = pd.DataFrame(data_store)
_env_dt = exo_env.base_env.unwrapped.dt


Exoskeleton env obs space: (9,)
Frozen policy obs space: (9,)
Number of muscles (MF values): 6
New wrapper obs space: (15,)
✓ Wrapper will extract first 9 dims for frozen policy
MA: [0.1176 0.0433 0.1065 0.14   0.0259 0.0218]
MR: [0.1166 0.069  0.0978 0.0256 0.0265 0.002 ]
MF: [0.7658 0.8877 0.7957 0.8344 0.9476 0.9762]
Ep 1 of 250
Ep 2 of 250
Ep 3 of 250
Ep 4 of 250
Ep 5 of 250
Ep 6 of 250
Ep 7 of 250
Ep 8 of 250
Ep 9 of 250
Ep 10 of 250
Ep 11 of 250
Ep 12 of 250
Ep 13 of 250
Ep 14 of 250
Ep 15 of 250
Ep 16 of 250
Ep 17 of 250
Ep 18 of 250
Ep 19 of 250
Ep 20 of 250
Ep 21 of 250
Ep 22 of 250
Ep 23 of 250
Ep 24 of 250
Ep 25 of 250
Ep 26 of 250
Ep 27 of 250
Ep 28 of 250
Ep 29 of 250
Ep 30 of 250
Ep 31 of 250
Ep 32 of 250
Ep 33 of 250
Ep 34 of 250
Ep 35 of 250
Ep 36 of 250
Ep 37 of 250
Ep 38 of 250
Ep 39 of 250
Ep 40 of 250
Ep 41 of 250
Ep 42 of 250
Ep 43 of 250
Ep 44 of 250
Ep 45 of 250
Ep 46 of 250
Ep 47 of 250
Ep 48 of 250
Ep 49 of 250
Ep 50 of 250
Ep 51 of 250
Ep 52 of 250
Ep 53 of 25

In [116]:
from stable_baselines3 import PPO
from myosuite.utils import gym

# ------------------------
# Training setup
# ------------------------
env_name = 'myoFatiElbowPose1D6MExoRandom-v0'
frozen_policy_path = 'ElbowPose_policy'

# Create base env
base_env = gym.make(env_name)

exo_env = ExoOnlyWrapper(base_env, frozen_policy_path=frozen_policy_path, smart_reset=True)

# Train PPO for phase 1
#ppo_exo = PPO.load('ExoOnly_PPO_policy_final', env = exo_env)
ppo_exo = PPO('MlpPolicy', exo_env, verbose=1)

ppo_exo.learn(total_timesteps=500000)


ppo_exo.save('ExoOnly_PPO_policy_final')

print('Final policy saved as: ExoOnly_PPO_policy_final')
print('Total training timesteps: 2 * 500,000')

Exoskeleton env obs space: (9,)
Frozen policy obs space: (9,)
Number of muscles (MF values): 6
New wrapper obs space: (15,)
✓ Wrapper will extract first 9 dims for frozen policy
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 100      |
|    ep_rew_mean     | -102     |
| time/              |          |
|    fps             | 473      |
|    iterations      | 1        |
|    time_elapsed    | 4        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 100         |
|    ep_rew_mean          | -103        |
| time/                   |             |
|    fps                  | 330         |
|    iterations           | 2           |
|    time_elapsed         | 12          |
|    total_timesteps      | 4096        |
| train/  

In [111]:
ppo_exo.save('ExoOnly_PPO_policy_final')


In [117]:
# Strictly data_preprocessing-style evaluation loop for ExoOnlyWrapper, only record first and last episode video
import myosuite
from myosuite.utils import gym
import skvideo.io
import numpy as np
import os
from stable_baselines3 import PPO
import matplotlib.pyplot as plt
import time
from IPython.display import HTML
from base64 import b64encode
import PIL.Image, PIL.ImageDraw

# ------------------------
# Utility: show video inline
# ------------------------
def show_video(video_path, video_width=400):
    video_file = open(video_path, "r+b").read()
    video_url = f'data:video/mp4;base64,{b64encode(video_file).decode()}'
    return HTML(f"<video autoplay width={video_width} controls><source src=\"{video_url}\"></video>")

def add_text_to_frame(frame, text, pos=(20, 20), color=(255, 0, 0), fontsize=12):
    if isinstance(frame, np.ndarray):
        frame = PIL.Image.fromarray(frame)
    draw = PIL.ImageDraw.Draw(frame)
    draw.text(pos, text, fill=color)
    return frame

# ------------------------
# Import custom wrapper
# ------------------------
from stable_baselines3 import PPO
import gymnasium as gym

class ExoOnlyWrapper(gym.Env):
    def __init__(self, base_env, frozen_policy_path, smart_reset=False):
        super(ExoOnlyWrapper, self).__init__()
        self.base_env = base_env
        self.frozen_policy = PPO.load(frozen_policy_path)
        self.action_space = gym.spaces.Box(low=np.array([0.0]), high=np.array([1.0]), dtype=np.float32)
        
        # Get number of muscles for MF observation space calculation
        self.n_muscles = len(self.base_env.unwrapped.muscle_fatigue.MA)
        
        # Store original observation space from the exoskeleton environment
        self.exo_obs_space = self.base_env.observation_space
        
        # The frozen policy was trained on the non-exoskeleton environment
        # We need to extract the base observations (first 9 dimensions)
        self.base_obs_dim = self.frozen_policy.observation_space.shape[0]
        self.exo_obs_dim = self.exo_obs_space.shape[0]
        
        print(f"Exoskeleton env obs space: {self.exo_obs_space.shape}")
        print(f"Frozen policy obs space: {self.frozen_policy.observation_space.shape}")
        print(f"Number of muscles (MF values): {self.n_muscles}")
        
        # Verify that exoskeleton environment has more observations than the frozen policy expects
        if self.exo_obs_dim < self.base_obs_dim:
            raise ValueError(
                f"Exoskeleton environment has fewer observations ({self.exo_obs_dim}) "
                f"than frozen policy expects ({self.base_obs_dim})"
            )
        
        # Create new observation space: base obs (9D) + MF values (6D) = 15D
        # But we want: base obs (9D) + MF values (6D) for the new policy
        base_obs_low = self.exo_obs_space.low[:self.base_obs_dim]
        base_obs_high = self.exo_obs_space.high[:self.base_obs_dim]
        
        # MF values are bounded between 0.0 and 1.0
        mf_low = np.zeros(self.n_muscles, dtype=np.float32)
        mf_high = np.ones(self.n_muscles, dtype=np.float32)
        
        # Combined observation space: base observations + MF values
        combined_low = np.concatenate([base_obs_low, mf_low])
        combined_high = np.concatenate([base_obs_high, mf_high])
        
        self.observation_space = gym.spaces.Box(
            low=combined_low,
            high=combined_high,
            dtype=np.float32
        )
        
        print(f"New wrapper obs space: {self.observation_space.shape}")
        print(f"✓ Wrapper will extract first {self.base_obs_dim} dims for frozen policy")
        
        self.smart_reset = smart_reset
        if smart_reset == False:
            self.base_env.unwrapped.set_fatigue_reset_random(True)

    def _get_extended_obs(self, exo_obs):
        """
        Extend base observation with muscle fatigue (MF) values
        Args:
            exo_obs: Full observation from exoskeleton environment (15D)
        Returns:
            Extended observation: base obs (9D) + MF values (6D) = 15D
        """
        # Extract base observations (first 9 dimensions) from exoskeleton observation
        base_obs = exo_obs[:self.base_obs_dim].astype(np.float32)
        
        # Get current MF values from the environment
        mf_values = self.base_env.unwrapped.muscle_fatigue.MF.copy().astype(np.float32)
        
        # Concatenate base observation with MF values
        extended_obs = np.concatenate([base_obs, mf_values])
        
        return extended_obs

    def _get_base_obs_for_policy(self, exo_obs):
        """
        Extract base observations for the frozen policy
        Args:
            exo_obs: Full observation from exoskeleton environment (15D)
        Returns:
            Base observation for frozen policy (9D)
        """
        return exo_obs[:self.base_obs_dim].astype(np.float32)

    def reset(self, **kwargs):
        exo_obs, info = self.base_env.reset(fatigue_reset=True)
        low = self.base_env.unwrapped.target_jnt_range[:, 0]
        high = self.base_env.unwrapped.target_jnt_range[:, 1]
        new_target = np.random.uniform(low, high)
        self.base_env.unwrapped.target_jnt_value = [1.7]
        self.base_env.unwrapped.target_type = 'fixed'

        if self.smart_reset:
            # Random fatigue per muscle in [0.7, 1.0]
            MF = np.random.uniform(0.99, 1.0, size=self.n_muscles)

            # For the remaining fraction (1 - MF), randomly split between MA and MR
            remaining = 1.0 - MF
            split = np.random.uniform(0.0, 1.0, size=self.n_muscles)  # split ratio
            MA = remaining * split
            MR = remaining * (1.0 - split)

            # Write back into environment
            self.base_env.unwrapped.muscle_fatigue.MA[:] = MA
            self.base_env.unwrapped.muscle_fatigue.MR[:] = MR
            self.base_env.unwrapped.muscle_fatigue.MF[:] = MF

        # Return extended observation (base + MF) for the new exoskeleton policy
        extended_obs = self._get_extended_obs(exo_obs)
        return extended_obs, info

    def step(self, exo_action):
        exo_action = np.array([exo_action], dtype=np.float32).reshape(-1)
        
        # Get the current observation from the exoskeleton environment (15D)
        try:
            current_exo_obs = self.base_env._get_obs()
        except AttributeError:
            current_exo_obs = self.base_env.unwrapped.get_obs()
        
        # Extract base observations (9D) for the frozen policy
        base_obs_for_policy = self._get_base_obs_for_policy(current_exo_obs)
        
        # Use the base observation for the frozen policy prediction
        muscle_actions, _ = self.frozen_policy.predict(base_obs_for_policy, deterministic=True)
        
        full_action = np.concatenate([exo_action, [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]])
        next_exo_obs, reward, done, truncated, info = self.base_env.step(full_action)
        
        # Return extended observation (base + MF) for the new exoskeleton policy
        extended_obs = self._get_extended_obs(next_exo_obs)
        return extended_obs, reward, done, truncated, info

    def render(self, *args, **kwargs):
        return self.base_env.render(*args, **kwargs)

    def close(self):
        self.base_env.close()

In [118]:
# ------------------------
# Evaluation setup
# ------------------------
env_name = "myoFatiElbowPose1D6MExoRandom-v0"
GENERATE_VIDEO = True

# Load base env and wrap it
base_env = gym.make(env_name)
exo_env = ExoOnlyWrapper(base_env, frozen_policy_path="ElbowPose_policy")

# Set fixed state for consistent evaluation
exo_env.base_env.unwrapped.set_fatigue_reset_random(False)
exo_env.base_env.reset()

max_steps = exo_env.base_env.spec.max_episode_steps
data_exo = []
data_passive = []
frames_exo = []
frames_passive = []

t_a = 1.7
# Fix target and fatigue values
low = exo_env.base_env.unwrapped.target_jnt_range[:, 0]
high = exo_env.base_env.unwrapped.target_jnt_range[:, 1]
# Set fixed target (you can change this value as needed)
exo_env.base_env.unwrapped.target_jnt_value = [t_a]
exo_env.base_env.unwrapped.target_type = 'fixed'
exo_env.base_env.unwrapped.update_target(restore_sim=True)

n_muscles = len(exo_env.base_env.unwrapped.muscle_fatigue.MA)
MF_fixed = np.random.uniform(0.9999999999999999999999999999999, 1.0, size=n_muscles)
remaining = 1.0 - MF_fixed
split = np.random.uniform(0.0, 1.0, size=n_muscles)
MA_fixed = remaining * split
MR_fixed = remaining * (1.0 - split)

# Set fixed fatigue values
exo_env.base_env.unwrapped.muscle_fatigue.MA[:] = MA_fixed
exo_env.base_env.unwrapped.muscle_fatigue.MR[:] = MR_fixed
exo_env.base_env.unwrapped.muscle_fatigue.MF[:] = MF_fixed

print("MA:", exo_env.base_env.unwrapped.muscle_fatigue.MA)
print("MR:", exo_env.base_env.unwrapped.muscle_fatigue.MR)
print("MF:", exo_env.base_env.unwrapped.muscle_fatigue.MF)

# Load trained exoskeleton policy
exo_policy = PPO.load("ExoOnly_PPO_policy_final")

# ------------------------
# First evaluation episode - Exoskeleton policy
# ------------------------
print("Running episode with exoskeleton assistance...")
obs, _ = exo_env.reset()
tr = 0
for _cstep in range(max_steps):
    if GENERATE_VIDEO:
        frame = exo_env.base_env.unwrapped.sim.renderer.render_offscreen(width=400, height=400, camera_id=0)
        _current_time = _cstep * exo_env.base_env.unwrapped.dt
        frame = np.array(add_text_to_frame(frame,
                f"t={str(int(_current_time//60)).zfill(2)}:{str(int(_current_time%60)).zfill(2)}min",
                pos=(285, 3), color=(0, 0, 0), fontsize=18))
        frames_exo.append(frame)
    
    # Use trained exoskeleton policy or random action

    exo_action, _ = exo_policy.predict(obs, deterministic=True)
    print(exo_action)


    next_obs, reward, done, truncated, info = exo_env.step([exo_action])
    tr += reward
    obs = next_obs
    
    data_exo.append({
        "exo_action": exo_action.copy(),
        "jpos": exo_env.base_env.unwrapped.sim.data.qpos.copy(),
        "mlen": exo_env.base_env.unwrapped.sim.data.actuator_length.copy(),
        "act": exo_env.base_env.unwrapped.sim.data.act.copy(),
        "reward": reward,
        "solved": exo_env.base_env.unwrapped.rwd_dict['solved'].item(),
        "pose_err": exo_env.base_env.unwrapped.get_obs_dict(exo_env.base_env.unwrapped.sim)["pose_err"],
        "MA": exo_env.base_env.unwrapped.muscle_fatigue.MA.copy(),
        "MR": exo_env.base_env.unwrapped.muscle_fatigue.MR.copy(),
        "MF": exo_env.base_env.unwrapped.muscle_fatigue.MF.copy(),
        "ctrl": exo_env.base_env.unwrapped.last_ctrl.copy()
    })
    
    if done or truncated:
        break

# ------------------------
# Reset to same initial conditions for second episode
# ------------------------
print("Total Reward:", tr)
print("Resetting to same initial conditions...")
# Reset the fatigue and target to the same initial values

# Reset environment to same initial state
obs, _ = exo_env.reset()

low = exo_env.base_env.unwrapped.target_jnt_range[:, 0]
high = exo_env.base_env.unwrapped.target_jnt_range[:, 1]
# Set fixed target (you can change this value as needed)
exo_env.base_env.unwrapped.target_type = 'fixed'
exo_env.base_env.unwrapped.target_jnt_value = [t_a]
exo_env.base_env.unwrapped.update_target(restore_sim=True)

exo_env.base_env.unwrapped.muscle_fatigue.MA[:] = MA_fixed
exo_env.base_env.unwrapped.muscle_fatigue.MR[:] = MR_fixed
exo_env.base_env.unwrapped.muscle_fatigue.MF[:] = MF_fixed

print("MA:", exo_env.base_env.unwrapped.muscle_fatigue.MA)
print("MR:", exo_env.base_env.unwrapped.muscle_fatigue.MR)
print("MF:", exo_env.base_env.unwrapped.muscle_fatigue.MF)
# ------------------------
# Second evaluation episode - Passive exoskeleton
# ------------------------
print("Running episode with passive exoskeleton (no assistance)...")


for _cstep in range(max_steps):
    if GENERATE_VIDEO:
        frame = exo_env.base_env.unwrapped.sim.renderer.render_offscreen(width=400, height=400, camera_id=0)
        _current_time = _cstep * exo_env.base_env.unwrapped.dt
        frame = np.array(add_text_to_frame(frame,
                f"t={str(int(_current_time//60)).zfill(2)}:{str(int(_current_time%60)).zfill(2)}min",
                pos=(285, 3), color=(0, 0, 0), fontsize=18))
        frames_passive.append(frame)
    
    # Passive exoskeleton (no assistance)
    exo_action = np.array([0.0], dtype=np.float32)
    next_obs, reward, done, truncated, info = exo_env.step(exo_action)
    obs = next_obs
    
    data_passive.append({
        "exo_action": exo_action.copy(),
        "jpos": exo_env.base_env.unwrapped.sim.data.qpos.copy(),
        "mlen": exo_env.base_env.unwrapped.sim.data.actuator_length.copy(),
        "act": exo_env.base_env.unwrapped.sim.data.act.copy(),
        "reward": reward,
        "solved": exo_env.base_env.unwrapped.rwd_dict['solved'].item(),
        "pose_err": exo_env.base_env.unwrapped.get_obs_dict(exo_env.base_env.unwrapped.sim)["pose_err"],
        "MA": exo_env.base_env.unwrapped.muscle_fatigue.MA.copy(),
        "MR": exo_env.base_env.unwrapped.muscle_fatigue.MR.copy(),
        "MF": exo_env.base_env.unwrapped.muscle_fatigue.MF.copy(),
        "ctrl": exo_env.base_env.unwrapped.last_ctrl.copy()
    })
    
    if done or truncated:
        break

exo_env.close()

# ------------------------
# Save & display videos
# ------------------------
if GENERATE_VIDEO and frames_exo:
    os.makedirs('videos', exist_ok=True)
    video_path_exo = 'videos/episode_exo_assisted.mp4'
    skvideo.io.vwrite(video_path_exo, np.asarray(frames_exo), outputdict={'-pix_fmt': 'yuv420p'})

if GENERATE_VIDEO and frames_passive:
    os.makedirs('videos', exist_ok=True)
    video_path_passive = 'videos/episode_passive.mp4'
    skvideo.io.vwrite(video_path_passive, np.asarray(frames_passive), outputdict={'-pix_fmt': 'yuv420p'})

if GENERATE_VIDEO and 'video_path_exo' in locals():
    display(show_video(video_path_exo))
if GENERATE_VIDEO and 'video_path_passive' in locals():
    display(show_video(video_path_passive))

# ------------------------
# Export logged data
# ------------------------
import pandas as pd
df_exo = pd.DataFrame(data_exo)
df_passive = pd.DataFrame(data_passive)
_env_dt = exo_env.base_env.unwrapped.dt

print(f"Exoskeleton assisted episode: {len(data_exo)} steps")
print(f"Passive exoskeleton episode: {len(data_passive)} steps")

Exoskeleton env obs space: (9,)
Frozen policy obs space: (9,)
Number of muscles (MF values): 6
New wrapper obs space: (15,)
✓ Wrapper will extract first 9 dims for frozen policy
MA: [0. 0. 0. 0. 0. 0.]
MR: [0. 0. 0. 0. 0. 0.]
MF: [1. 1. 1. 1. 1. 1.]
Running episode with exoskeleton assistance...
[0.91]
[0.9166]
[0.9116]
[0.9019]
[0.8882]
[0.8709]
[0.8505]
[0.8277]
[0.803]
[0.7773]
[0.7513]
[0.726]
[0.702]
[0.68]
[0.6605]
[0.6438]
[0.6302]
[0.6197]
[0.6123]
[0.6081]
[0.6067]
[0.608]
[0.6117]
[0.6174]
[0.6247]
[0.6332]
[0.6426]
[0.6523]
[0.6621]
[0.6715]
[0.6804]
[0.6884]
[0.6954]
[0.7012]
[0.7058]
[0.7092]
[0.7114]
[0.7125]
[0.7125]
[0.7115]
[0.7098]
[0.7073]
[0.7043]
[0.701]
[0.6974]
[0.6937]
[0.6901]
[0.6865]
[0.6833]
[0.6803]
[0.6777]
[0.6755]
[0.6738]
[0.6725]
[0.6717]
[0.6713]
[0.6713]
[0.6717]
[0.6724]
[0.6733]
[0.6745]
[0.6757]
[0.6771]
[0.6785]
[0.6799]
[0.6812]
[0.6825]
[0.6836]
[0.6846]
[0.6854]
[0.686]
[0.6865]
[0.6868]
[0.687]
[0.687]
[0.6868]
[0.6866]
[0.6863]
[0.6859]
[0.6

Exoskeleton assisted episode: 100 steps
Passive exoskeleton episode: 100 steps


In [119]:
#print(f"\nData integrity check:")
print(f"Exo actions range: {df_exo['exo_action'].apply(lambda x: x[0]).min():.3f} to {df_exo['exo_action'].apply(lambda x: x[0]).max():.3f}")
print(f"Passive actions range: {df_passive['exo_action'].apply(lambda x: x[0]).min():.3f} to {df_passive['exo_action'].apply(lambda x: x[0]).max():.3f}")
print(f"Exo final MF: {data_exo[-1]['MF']}")
print(f"Passive final MF: {data_passive[-1]['MF']}")

# ------------------------
# CORRECTED ANALYSIS FUNCTIONS
# ------------------------

def extract_data_arrays_corrected(df):
    """Extract time series data from dataframe with proper error handling"""
    data_store = df.to_dict('records')
    
    # Calculate average fatigue (MF) across all muscles at each timestep
    avg_fatigue = np.array([np.mean(d['MF']) for d in data_store])
    
    # Extract pose error magnitude
    pose_error = np.array([np.linalg.norm(d['pose_err']) for d in data_store])
    
    # Extract rewards
    rewards = np.array([d['reward'] for d in data_store])
    
    # Extract solved status
    solved = np.array([d['solved'] for d in data_store])
    
    # Extract time stamps
    time_stamps = np.array([d['time'] for d in data_store])
    
    return avg_fatigue, pose_error, rewards, solved, time_stamps, data_store

# Extract data for both conditions
exo_fatigue, exo_pose_err, exo_rewards, exo_solved, time_exo, exo_data = extract_data_arrays_corrected(df_exo)
passive_fatigue, passive_pose_err, passive_rewards, passive_solved, time_passive, passive_data = extract_data_arrays_corrected(df_passive)

# Extract exoskeleton actions properly
exo_actions = np.array([a[0] if isinstance(a, (list, np.ndarray)) else a for a in df_exo["exo_action"]])
passive_actions = np.array([a[0] if isinstance(a, (list, np.ndarray)) else a for a in df_passive["exo_action"]])

print(f"\nAction verification:")
print(f"Exo actions non-zero count: {np.count_nonzero(exo_actions)}/{len(exo_actions)}")
print(f"Passive actions non-zero count: {np.count_nonzero(passive_actions)}/{len(passive_actions)}")

# ------------------------
# CORRECTED VISUALIZATIONS
# ------------------------

# 1. Fixed Average Fatigue Comparison
plt.figure(figsize=(12, 6))
plt.plot(time_exo, exo_fatigue, 'b-', linewidth=2, label='Exo-Assisted', alpha=0.8)
plt.plot(time_passive, passive_fatigue, 'r-', linewidth=2, label='Passive/No Assistance', alpha=0.8)

# Add filled areas with transparency
plt.fill_between(time_exo, 0, exo_fatigue, alpha=0.3, color='blue', label='Exo Area')
plt.fill_between(time_passive, 0, passive_fatigue, alpha=0.3, color='red', label='Passive Area')

plt.title('Average Muscle Fatigue Level Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Time (s)')
plt.ylabel('Average Fatigue Level (MF)')
plt.legend()
plt.grid(True, alpha=0.3)

# Add summary statistics
exo_final_fatigue = exo_fatigue[-1]
passive_final_fatigue = passive_fatigue[-1]
fatigue_reduction = ((passive_final_fatigue - exo_final_fatigue) / passive_final_fatigue) * 100 if passive_final_fatigue > 0 else 0

plt.text(0.02, 0.98, f'Final Fatigue - Exo: {exo_final_fatigue:.3f}\n' +
                      f'Final Fatigue - Passive: {passive_final_fatigue:.3f}\n' +
                      f'Fatigue Reduction: {fatigue_reduction:.1f}%\n' +
                      f'Reward Diff: {tr_exo - tr_passive:.1f}',
         transform=plt.gca().transAxes, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
plt.tight_layout()
plt.show()

# 2. Exoskeleton Action vs Fatigue Correlation Analysis
from scipy.stats import pearsonr

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Top subplot: Time series with dual y-axes
ax1_twin = ax1.twinx()

# Plot exo action and fatigue on separate y-axes but same time axis
line1 = ax1.plot(time_exo, exo_actions, 'g-', linewidth=2, label='Exo Action', alpha=0.8)
line2 = ax1_twin.plot(time_exo, exo_fatigue, 'orange', linewidth=2, label='Avg Fatigue', alpha=0.8)

ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Exoskeleton Action', color='g')
ax1_twin.set_ylabel('Average Fatigue Level', color='orange')
ax1.set_title('Exoskeleton Action vs Average Fatigue Over Time', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Calculate correlation
correlation, p_value = pearsonr(exo_actions, exo_fatigue)

# Add correlation info
ax1.text(0.02, 0.98, f'Correlation: {correlation:.3f}\np-value: {p_value:.4f}\n' +
         f'Action Range: {exo_actions.min():.3f} to {exo_actions.max():.3f}\n' +
         f'Fatigue Range: {exo_fatigue.min():.3f} to {exo_fatigue.max():.3f}',
         transform=ax1.transAxes, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

# Bottom subplot: Scatter plot of action vs fatigue
scatter = ax2.scatter(exo_fatigue, exo_actions, alpha=0.6, c=time_exo, cmap='viridis', s=30)
ax2.set_xlabel('Average Fatigue Level')
ax2.set_ylabel('Exoskeleton Action')
ax2.set_title('Exoskeleton Action Response to Fatigue Level', fontsize=12)
ax2.grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(exo_fatigue, exo_actions, 1)
p = np.poly1d(z)
ax2.plot(exo_fatigue, p(exo_fatigue), "r--", alpha=0.8, linewidth=2, 
         label=f'Trend: y = {z[0]:.3f}x + {z[1]:.3f}')
ax2.legend()

colorbar = plt.colorbar(scatter, ax=ax2)
colorbar.set_label('Time (s)')

plt.tight_layout()
plt.show()

# 3. Corrected Success Rate Analysis  
plt.figure(figsize=(12, 6))

# Calculate RUNNING success rate (cumulative)
exo_success_cumulative = np.cumsum(exo_solved)
passive_success_cumulative = np.cumsum(passive_solved)

# Success rate as percentage of total steps
exo_success_rate = exo_success_cumulative / len(exo_solved) * 100
passive_success_rate = passive_success_cumulative / len(passive_solved) * 100

plt.plot(time_exo, exo_success_rate, 'b-', linewidth=2, label='Exo-Assisted', alpha=0.8)
plt.plot(time_passive, passive_success_rate, 'r-', linewidth=2, label='Passive/No Assistance', alpha=0.8)
plt.title('Cumulative Success Rate Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Time (s)')
plt.ylabel('Cumulative Success Rate (%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, max(max(exo_success_rate), max(passive_success_rate)) + 5)

plt.text(0.02, 0.98, f'Final Success Rate - Exo: {exo_success_rate[-1]:.1f}%\n' +
                      f'Final Success Rate - Passive: {passive_success_rate[-1]:.1f}%\n' +
                      f'Total Successes - Exo: {int(exo_success_cumulative[-1])}\n' +
                      f'Total Successes - Passive: {int(passive_success_cumulative[-1])}',
         transform=plt.gca().transAxes, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
plt.tight_layout()
plt.show()

# 4. Pose Error Comparison - Direct Task Performance
plt.figure(figsize=(12, 6))
plt.plot(time_exo, exo_pose_err, 'b-', linewidth=2, label='Exo-Assisted', alpha=0.8)
plt.plot(time_passive, passive_pose_err, 'r-', linewidth=2, label='Passive/No Assistance', alpha=0.8)
plt.fill_between(time_exo, exo_pose_err, alpha=0.3, color='blue')
plt.fill_between(time_passive, passive_pose_err, alpha=0.3, color='red')
plt.title('Pose Error Comparison - Task Performance', fontsize=14, fontweight='bold')
plt.xlabel('Time (s)')
plt.ylabel('Pose Error Magnitude')
plt.legend()
plt.grid(True, alpha=0.3)

# Calculate and display statistics
exo_avg_error = np.mean(exo_pose_err)
passive_avg_error = np.mean(passive_pose_err)
exo_std_error = np.std(exo_pose_err)
passive_std_error = np.std(passive_pose_err)
error_improvement = ((passive_avg_error - exo_avg_error) / passive_avg_error) * 100 if passive_avg_error > 0 else 0

plt.text(0.02, 0.98, f'Avg Error - Exo: {exo_avg_error:.4f} ± {exo_std_error:.4f}\n' +
                      f'Avg Error - Passive: {passive_avg_error:.4f} ± {passive_std_error:.4f}\n' +
                      f'Error Reduction: {error_improvement:.1f}%\n' +
                      f'Final Error - Exo: {exo_pose_err[-1]:.4f}\n' +
                      f'Final Error - Passive: {passive_pose_err[-1]:.4f}',
         transform=plt.gca().transAxes, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
plt.tight_layout()
plt.show()

# 5. Cumulative Reward Comparison - Learning Effectiveness
plt.figure(figsize=(12, 6))
exo_cumulative = np.cumsum(exo_rewards)
passive_cumulative = np.cumsum(passive_rewards)

plt.plot(time_exo, exo_cumulative, 'b-', linewidth=2, label='Exo-Assisted', alpha=0.8)
plt.plot(time_passive, passive_cumulative, 'r-', linewidth=2, label='Passive/No Assistance', alpha=0.8)
plt.title('Cumulative Reward Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Time (s)')
plt.ylabel('Cumulative Reward')
plt.legend()
plt.grid(True, alpha=0.3)

# Add reward rate information
exo_reward_rate = exo_cumulative[-1] / len(exo_rewards) if len(exo_rewards) > 0 else 0
passive_reward_rate = passive_cumulative[-1] / len(passive_rewards) if len(passive_rewards) > 0 else 0
total_reward_improvement = exo_cumulative[-1] - passive_cumulative[-1]

plt.text(0.02, 0.98, f'Total Reward - Exo: {exo_cumulative[-1]:.2f}\n' +
                      f'Total Reward - Passive: {passive_cumulative[-1]:.2f}\n' +
                      f'Improvement: {total_reward_improvement:.2f}\n' +
                      f'Avg Reward/Step - Exo: {exo_reward_rate:.3f}\n' +
                      f'Avg Reward/Step - Passive: {passive_reward_rate:.3f}',
         transform=plt.gca().transAxes, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
plt.tight_layout()
plt.show()

# 6. Action Efficiency Analysis - How much action is needed
plt.figure(figsize=(12, 6))

# Calculate moving average of exo actions to see efficiency patterns
window_size = max(1, len(exo_actions) // 20)  # 5% window
exo_action_smooth = np.convolve(exo_actions, np.ones(window_size)/window_size, mode='same')

plt.plot(time_exo, exo_actions, 'g-', alpha=0.5, linewidth=1, label='Raw Exo Action')
plt.plot(time_exo, exo_action_smooth, 'darkgreen', linewidth=2, label='Smoothed Exo Action')
plt.axhline(y=np.mean(exo_actions), color='red', linestyle='--', alpha=0.7, label=f'Mean Action: {np.mean(exo_actions):.3f}')

plt.title('Exoskeleton Action Efficiency Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Time (s)')
plt.ylabel('Exoskeleton Action')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 1.05)

# Action statistics
action_variance = np.var(exo_actions)
action_changes = np.sum(np.abs(np.diff(exo_actions)))
action_efficiency = np.mean(exo_actions) / (action_variance + 1e-8)  # Avoid division by zero

plt.text(0.02, 0.98, f'Action Statistics:\n' +
                      f'Mean: {np.mean(exo_actions):.3f}\n' +
                      f'Std: {np.std(exo_actions):.3f}\n' +
                      f'Max: {np.max(exo_actions):.3f}\n' +
                      f'Total Action Changes: {action_changes:.2f}\n' +
                      f'Action Efficiency: {action_efficiency:.2f}',
         transform=plt.gca().transAxes, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
plt.tight_layout()
plt.show()

# 7. Performance Stability Analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Left: Error stability over time
ax1.plot(time_exo, exo_pose_err, 'b-', alpha=0.7, label='Exo-Assisted')
ax1.plot(time_passive, passive_pose_err, 'r-', alpha=0.7, label='Passive')
ax1.fill_between(time_exo, exo_pose_err - exo_std_error, exo_pose_err + exo_std_error, 
                 alpha=0.2, color='blue', label='Exo ± 1σ')
ax1.fill_between(time_passive, passive_pose_err - passive_std_error, passive_pose_err + passive_std_error, 
                 alpha=0.2, color='red', label='Passive ± 1σ')
ax1.set_title('Error Stability Comparison')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Pose Error')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: Reward stability
window_reward_exo = np.convolve(exo_rewards, np.ones(max(1, len(exo_rewards)//10))/max(1, len(exo_rewards)//10), mode='same')
window_reward_passive = np.convolve(passive_rewards, np.ones(max(1, len(passive_rewards)//10))/max(1, len(passive_rewards)//10), mode='same')

ax2.plot(time_exo, window_reward_exo, 'b-', linewidth=2, label='Exo-Assisted (smoothed)')
ax2.plot(time_passive, window_reward_passive, 'r-', linewidth=2, label='Passive (smoothed)')
ax2.axhline(y=np.mean(exo_rewards), color='blue', linestyle='--', alpha=0.7, label=f'Exo Mean: {np.mean(exo_rewards):.3f}')
ax2.axhline(y=np.mean(passive_rewards), color='red', linestyle='--', alpha=0.7, label=f'Passive Mean: {np.mean(passive_rewards):.3f}')
ax2.set_title('Reward Stability Comparison')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Smoothed Reward')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("COMPREHENSIVE POLICY EVALUATION SUMMARY")
print("="*60)
print(f"Episode lengths: Exo={len(data_exo)}, Passive={len(data_passive)}")
print(f"Total rewards: Exo={tr_exo:.2f}, Passive={tr_passive:.2f}")
print(f"Reward improvement: {tr_exo - tr_passive:.2f} ({((tr_exo - tr_passive)/abs(tr_passive)*100 if tr_passive != 0 else 0):.1f}%)")
print(f"Fatigue reduction: {fatigue_reduction:.1f}%")
print(f"Pose error reduction: {error_improvement:.1f}%")
print(f"Success improvement: {exo_success_rate[-1] - passive_success_rate[-1]:.1f}%")
print(f"Action-Fatigue correlation: {correlation:.3f} (p={p_value:.4f})")
print(f"Action efficiency: Mean={np.mean(exo_actions):.3f}, Std={np.std(exo_actions):.3f}")

# Policy effectiveness assessment
print("\n" + "-"*40)
print("POLICY EFFECTIVENESS ASSESSMENT:")
print("-"*40)

if tr_exo > tr_passive:
    print("✓ Policy improves total reward")
else:
    print("✗ Policy decreases total reward")

if error_improvement > 0:
    print("✓ Policy reduces pose error")
else:
    print("✗ Policy increases pose error")

if fatigue_reduction > 0:
    print("✓ Policy reduces muscle fatigue")
else:
    print("✗ Policy increases muscle fatigue")

if correlation > 0.3:
    print("✓ Policy shows positive correlation with fatigue (responsive assistance)")
elif correlation < -0.3:
    print("? Policy shows negative correlation with fatigue (unusual)")
else:
    print("○ Policy shows weak correlation with fatigue (may not be fatigue-responsive)")

if np.std(exo_actions) < 0.3:
    print("✓ Policy actions are stable")
else:
    print("? Policy actions are highly variable")

print("="*60)

Exo actions range: 0.607 to 0.917
Passive actions range: 0.000 to 0.000
Exo final MF: [0.0014 0.0014 0.0014 0.0014 0.0014 0.0014]
Passive final MF: [0.9998 0.9998 0.9998 0.9998 0.9998 0.9998]


KeyError: 'time'